In [ ]:
import pandas as pd
import numpy as np

def engenharia_de_stints(csv_entrada, csv_saida):
    print("Analisando telemetria: Identificando Stints, In-laps e Safety Cars")

    # 1. Carrega a base unificada que você gerou
    df = pd.read_csv(csv_entrada, sep=';', decimal=',')

    # Garante que os dados estão ordenados por Carro e por Volta cronologicamente
    df['Lap_Int'] = df['Lap'].astype(int)
    df = df.sort_values(by=['Carro', 'Lap_Int']).reset_index(drop=True)

    # 2. Identificar In-Laps (Voltas com tempo de Pit Stop registrado)
    df['In_Lap'] = df['Pit Time'].notna()

    # 3. Criar a coluna de Stint
    # O Stint muda na volta SEGUINTE ao pit stop (na Out-Lap)
    # O .astype(int) transforma False em 0 e True em 1, permitindo a soma matemática
    df['Mudanca_Stint'] = df.groupby('Carro')['In_Lap'].shift(1).fillna(False).astype(int)
    df['Stint'] = df.groupby('Carro')['Mudanca_Stint'].cumsum() + 1

    # 4. Classificar o Tipo de Volta
    df['Tipo_Volta'] = 'Push' # Volta normal de aceleração
    
    # Marca as In-Laps
    df.loc[df['In_Lap'], 'Tipo_Volta'] = 'In-Lap'
    
    # Marca as Out-Laps (a volta que iniciou o novo stint, exceto a volta 1 da corrida)
    out_lap_mask = df.groupby('Carro')['In_Lap'].shift(1).fillna(False)
    df.loc[out_lap_mask, 'Tipo_Volta'] = 'Out-Lap'

    # 5. Detecção de Outliers (Safety Car, Tráfego, Erros)
    df['Outlier'] = False
    
    # In-Laps e Out-Laps são sempre outliers para cálculo de desgaste puro
    df.loc[df['Tipo_Volta'] != 'Push', 'Outlier'] = True

    # Calcula a mediana do tempo de volta APENAS das voltas 'Push' de cada Stint por Carro
    df_push = df[df['Tipo_Volta'] == 'Push']
    medias_stint = df_push.groupby(['Carro', 'Stint'])['Lap Tm (Segundos)'].transform('median')
    
    # Mapeia essas medianas de volta para o dataframe original
    df.loc[df['Tipo_Volta'] == 'Push', 'Tempo_Referencia_Stint'] = medias_stint
    
    # Regra: Se a volta for 5% mais lenta que a mediana do stint, é anomalia (SC ou erro)
    limite_tempo = df['Tempo_Referencia_Stint'] * 1.05
    mascara_lenta = (df['Tipo_Volta'] == 'Push') & (df['Lap Tm (Segundos)'] > limite_tempo)
    df.loc[mascara_lenta, 'Outlier'] = True

    # Limpeza final das colunas de apoio
    df = df.drop(columns=['Lap_Int', 'In_Lap', 'Mudanca_Stint', 'Tempo_Referencia_Stint'])

    # Salva a base preparada para a Inteligência Estratégica
    df.to_csv(csv_saida, index=False, sep=';', decimal=',')
    print(f"Engenharia de Stints concluída! Base pronta salva em: {csv_saida}")

# --- ÁREA DE EXECUÇÃO ---
arquivo_telemetria = '../data/03_processed/TELEMETRIA_FINAL_P1.csv'
arquivo_estrategia = '../data/03_processed/TELEMETRIA_ESTRATEGIA_P1.csv'

engenharia_de_stints(arquivo_telemetria, arquivo_estrategia)

🧠 Analisando telemetria: Identificando Stints, In-laps e Safety Cars...
📊 Engenharia de Stints concluída! Base pronta salva em: ../data/03_processed/TELEMETRIA_ESTRATEGIA_P1.csv
